In [1]:
# =========================================================
# INSTALLS (COLAB)
# =========================================================

!pip install -q timm datasets accelerate seaborn


In [3]:

# =========================================================
# IMPORTS
# =========================================================

import os
import json
import torch
import timm
import random
import numpy as np
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim

from tqdm.auto import tqdm

from datasets import load_dataset

from torchvision import transforms
from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    precision_recall_fscore_support
)

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if torch.cuda.is_available():

    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.backends.cudnn.benchmark = True

# =========================================================
# SEED
# =========================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed()

# =========================================================
# CONFIG
# =========================================================

class Config:

    IMAGE_SIZE = 300

    BATCH_SIZE = 32

    LEARNING_RATE = 3e-4

    FINE_TUNE_LR = 1e-5

    EPOCHS = 5

    FINE_TUNE_EPOCHS = 3

    NUM_WORKERS = 0

    MODEL_NAME = "efficientnet_b3"

    CHECKPOINT_DIR = "checkpoints"

    RESULTS_DIR = "results"

# =========================================================
# TRANSFORMS
# =========================================================

train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        Config.IMAGE_SIZE
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),

    transforms.RandomAffine(
        degrees=15,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([

    transforms.Resize(
        (Config.IMAGE_SIZE, Config.IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# =========================================================
# LOAD DATASET
# =========================================================

print("\nLoading PlantVillage dataset...")

dataset = load_dataset(
    "dpdl-benchmark/plant_village"
)

dataset = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

dataset["validation"] = dataset["test"]

del dataset["test"]

print(dataset)

# =========================================================
# CLASS NAMES
# =========================================================

class_names = [

    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",

    "Blueberry___healthy",

    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",

    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",

    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",

    "Orange___Haunglongbing_(Citrus_greening)",

    "Peach___Bacterial_spot",
    "Peach___healthy",

    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",

    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",

    "Raspberry___healthy",

    "Soybean___healthy",

    "Squash___Powdery_mildew",

    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",

    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

with open("classes.json", "w") as f:

    json.dump(class_names, f)

# =========================================================
# DATASET CLASS
# =========================================================

class PlantDataset(Dataset):

    def __init__(self, dataset, transform):

        self.dataset = dataset

        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        image = item["image"].convert("RGB")

        label = item["label"]

        image = self.transform(image)

        return image, torch.tensor(
            label,
            dtype=torch.long
        )

# =========================================================
# DATALOADERS
# =========================================================

train_dataset = PlantDataset(
    dataset["train"],
    train_transform
)

val_dataset = PlantDataset(
    dataset["validation"],
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=Config.NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=Config.NUM_WORKERS
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# =========================================================
# MODEL
# =========================================================

print("\nCreating EfficientNet-B3...")

model = timm.create_model(
    Config.MODEL_NAME,
    pretrained=True,
    num_classes=len(class_names)
)

model = model.to(device)

# =========================================================
# FREEZE BACKBONE
# =========================================================

def freeze_backbone(model):

    for param in model.parameters():

        param.requires_grad = False

    for param in model.classifier.parameters():

        param.requires_grad = True

# =========================================================
# UNFREEZE ALL
# =========================================================

def unfreeze_all(model):

    for param in model.parameters():

        param.requires_grad = True

# =========================================================
# TRAIN EPOCH
# =========================================================

def train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    epoch,
    total_epochs
):

    model.train()

    total_loss = 0

    correct = 0

    total = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{total_epochs}"
    )

    for images, labels in pbar:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    epoch_loss = total_loss / len(loader)

    epoch_acc = correct / total

    return epoch_loss, epoch_acc

# =========================================================
# VALIDATE
# =========================================================

def validate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0

    correct = 0

    total = 0

    all_preds = []

    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    epoch_loss = total_loss / len(loader)

    epoch_acc = correct / total

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_acc,
        precision,
        recall,
        f1
    )

# =========================================================
# TRAIN FUNCTION
# =========================================================

def train(
    model,
    train_loader,
    val_loader,
    num_epochs,
    learning_rate,
    phase_name
):

    print(f"\nStarting {phase_name}")

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=learning_rate
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )

    history = {

        'train_loss': [],
        'train_acc': [],

        'val_loss': [],
        'val_acc': [],

        'precision': [],
        'recall': [],
        'f1_score': []
    }

    best_acc = 0

    for epoch in range(num_epochs):

        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            epoch,
            num_epochs
        )

        (
            val_loss,
            val_acc,
            precision,
            recall,
            f1
        ) = validate(
            model,
            val_loader,
            criterion
        )

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)

        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        history['precision'].append(precision)
        history['recall'].append(recall)
        history['f1_score'].append(f1)

        print(f"\n{phase_name} Epoch {epoch+1}")

        print(f"Train Acc: {train_acc:.4f}")
        print(f"Val Acc:   {val_acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")

        if val_acc > best_acc:

            best_acc = val_acc

            os.makedirs(
                Config.CHECKPOINT_DIR,
                exist_ok=True
            )

            torch.save(
                model.state_dict(),
                os.path.join(
                    Config.CHECKPOINT_DIR,
                    f"best_{phase_name}.pth"
                )
            )

    return history

# =========================================================
# PHASE 1
# =========================================================

print("\nPHASE 1 - TRANSFER LEARNING")

freeze_backbone(model)

history_phase1 = train(
    model,
    train_loader,
    val_loader,
    num_epochs=Config.EPOCHS,
    learning_rate=Config.LEARNING_RATE,
    phase_name="transfer_learning"
)

# =========================================================
# SAVE PHASE 1
# =========================================================

os.makedirs(
    Config.RESULTS_DIR,
    exist_ok=True
)

with open(
    os.path.join(
        Config.RESULTS_DIR,
        "phase1_history.json"
    ),
    "w"
) as f:

    json.dump(
        history_phase1,
        f,
        indent=4
    )

# =========================================================
# PHASE 2
# =========================================================

print("\nPHASE 2 - FINE TUNING")

unfreeze_all(model)

history_phase2 = train(
    model,
    train_loader,
    val_loader,
    num_epochs=Config.FINE_TUNE_EPOCHS,
    learning_rate=Config.FINE_TUNE_LR,
    phase_name="fine_tuning"
)

# =========================================================
# SAVE PHASE 2
# =========================================================

with open(
    os.path.join(
        Config.RESULTS_DIR,
        "phase2_history.json"
    ),
    "w"
) as f:

    json.dump(
        history_phase2,
        f,
        indent=4
    )

# =========================================================
# SAVE FINAL MODEL
# =========================================================

torch.save(
    model.state_dict(),
    "efficientnet_b3_final.pth"
)

print("\nFinal model saved")

# =========================================================
# COMBINE HISTORY
# =========================================================

history = {

    'train_loss':
        history_phase1['train_loss']
        + history_phase2['train_loss'],

    'train_acc':
        history_phase1['train_acc']
        + history_phase2['train_acc'],

    'val_loss':
        history_phase1['val_loss']
        + history_phase2['val_loss'],

    'val_acc':
        history_phase1['val_acc']
        + history_phase2['val_acc'],

    'precision':
        history_phase1['precision']
        + history_phase2['precision'],

    'recall':
        history_phase1['recall']
        + history_phase2['recall'],

    'f1_score':
        history_phase1['f1_score']
        + history_phase2['f1_score']
}

# =========================================================
# SAVE HISTORY
# =========================================================

with open(

    os.path.join(
        Config.RESULTS_DIR,
        "training_history_convnextv2.json"
    ),

    "w"

) as f:

    json.dump(
        history,
        f,
        indent=4
    )

print("\nTraining complete")

Device: cuda
GPU: Tesla T4

Loading PlantVillage dataset...
DatasetDict({
    train: Dataset({
        features: ['image', 'image/filename', 'label'],
        num_rows: 43442
    })
    validation: Dataset({
        features: ['image', 'image/filename', 'label'],
        num_rows: 10861
    })
})

Train batches: 1358
Validation batches: 340

Creating EfficientNet-B3...

PHASE 1 - TRANSFER LEARNING

Starting transfer_learning


Epoch 1/5:   0%|          | 0/1358 [00:00<?, ?it/s]


transfer_learning Epoch 1
Train Acc: 0.5053
Val Acc:   0.7321
F1 Score:  0.7242


Epoch 2/5:   0%|          | 0/1358 [00:00<?, ?it/s]


transfer_learning Epoch 2
Train Acc: 0.7161
Val Acc:   0.8124
F1 Score:  0.8103


...


fine_tuning Epoch 2
Train Acc: 0.9007
Val Acc:   0.9576
F1 Score:  0.9580


Epoch 3/5:   0%|          | 0/1358 [00:00<?, ?it/s]


fine_tuning Epoch 3
Train Acc: 0.9212
Val Acc:   0.9682
F1 Score:  0.9682


Epoch 4/5:   0%|          | 0/1358 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
# =========================================================
# INSTALLS (COLAB)
# =========================================================

!pip install -q timm datasets seaborn

# =========================================================
# IMPORTS
# =========================================================

import os
import json
import torch
import timm
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from datasets import load_dataset

from torchvision import transforms
from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support
)

# =========================================================
# CONFIG
# =========================================================

class Config:

    MODEL_PATH = "/content/efficientnet_b3_final.pth"

    CLASSES_PATH = "/content/classes.json"

    RESULTS_DIR = "/content/evaluation_results"

    TRAINING_RESULTS = os.path.join(
        "/content/results",
        "training_history_convnextv2.json"
    )

    PHASE1_HISTORY = os.path.join(
        "/content/results",
        "phase1_history.json"
    )

    PHASE2_HISTORY = os.path.join(
        "/content/results",
        "phase2_history.json"
    )

    MODEL_NAME = "efficientnet_b3"

    IMAGE_SIZE = 300

    BATCH_SIZE = 32

    NUM_WORKERS = 0

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if torch.cuda.is_available():

    print(f"GPU: {torch.cuda.get_device_name(0)}")

# =========================================================
# TRANSFORMS
# =========================================================

val_transform = transforms.Compose([

    transforms.Resize(
        (Config.IMAGE_SIZE, Config.IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================================================
# DATASET
# =========================================================

class PlantDataset(Dataset):

    def __init__(self, dataset, transform):

        self.dataset = dataset

        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        image = item["image"].convert("RGB")

        label = item["label"]

        image = self.transform(image)

        return image, label

# =========================================================
# LOAD CLASSES
# =========================================================

def load_classes():

    with open(Config.CLASSES_PATH, "r") as f:

        class_names = json.load(f)

    print(f"Loaded classes: {len(class_names)}")

    return class_names

# =========================================================
# LOAD DATASET
# =========================================================

def load_validation_loader():

    print("\nLoading dataset...")

    dataset = load_dataset(
        "dpdl-benchmark/plant_village"
    )

    dataset = dataset["train"].train_test_split(
        test_size=0.2,
        seed=42
    )

    dataset["validation"] = dataset["test"]

    del dataset["test"]

    val_dataset = PlantDataset(
        dataset["validation"],
        val_transform
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS
    )

    print(f"Validation samples: {len(val_dataset)}")

    return val_loader

# =========================================================
# LOAD MODEL
# =========================================================

def load_model(num_classes):

    print("\nLoading model...")

    model = timm.create_model(
        Config.MODEL_NAME,
        pretrained=False,
        num_classes=num_classes
    )

    state_dict = torch.load(
        Config.MODEL_PATH,
        map_location=device
    )

    model.load_state_dict(state_dict)

    model.to(device)

    model.eval()

    print("Model loaded successfully")

    return model

# =========================================================
# RUN INFERENCE
# =========================================================

def run_inference(model, loader):

    all_preds = []

    all_labels = []

    print("\nRunning inference...")

    with torch.no_grad():

        pbar = tqdm(
            loader,
            desc="Evaluating"
        )

        for images, labels in pbar:

            images = images.to(device)

            outputs = model(images)

            probs = torch.softmax(
                outputs,
                dim=1
            )

            preds = probs.argmax(dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    return all_preds, all_labels

# =========================================================
# CALCULATE METRICS
# =========================================================

def calculate_metrics(all_labels, all_preds, class_names):

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted"
    )

    conf_matrix = confusion_matrix(
        all_labels,
        all_preds
    )

    report = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        output_dict=True
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "conf_matrix": conf_matrix,
        "report": report
    }

# =========================================================
# LOAD HISTORIES
# =========================================================

def load_histories():

    with open(Config.TRAINING_RESULTS, "r") as f:

        history = json.load(f)

    with open(Config.PHASE1_HISTORY, "r") as f:

        phase1_history = json.load(f)

    with open(Config.PHASE2_HISTORY, "r") as f:

        phase2_history = json.load(f)

    return history, phase1_history, phase2_history

# =========================================================
# MAIN DASHBOARD
# =========================================================

def generate_dashboard(metrics, history):

    fig = plt.figure(figsize=(18, 12))

    # ACCURACY

    plt.subplot(2,3,1)

    plt.plot(history["train_acc"], label="Train")
    plt.plot(history["val_acc"], label="Validation")

    plt.title("Accuracy")

    plt.legend()

    plt.grid(True)

    # LOSS

    plt.subplot(2,3,2)

    plt.plot(history["train_loss"], label="Train")
    plt.plot(history["val_loss"], label="Validation")

    plt.title("Loss")

    plt.legend()

    plt.grid(True)

    # METRICS

    plt.subplot(2,3,3)

    plt.plot(history["precision"], label="Precision")
    plt.plot(history["recall"], label="Recall")
    plt.plot(history["f1_score"], label="F1")

    plt.title("Metrics")

    plt.legend()

    plt.grid(True)

    # CONFUSION MATRIX

    plt.subplot(2,3,4)

    sns.heatmap(
        metrics["conf_matrix"],
        cmap="Blues"
    )

    plt.title("Confusion Matrix")

    # OVERALL

    plt.subplot(2,3,5)

    plt.axis("off")

    text = f"""

Accuracy : {metrics['accuracy']:.4f}

Precision: {metrics['precision']:.4f}

Recall   : {metrics['recall']:.4f}

F1-Score : {metrics['f1']:.4f}

"""

    plt.text(
        0.1,
        0.5,
        text,
        fontsize=16
    )

    plt.title("Overall Metrics")

    plt.tight_layout()

    os.makedirs(
        Config.RESULTS_DIR,
        exist_ok=True
    )

    save_path = os.path.join(
        Config.RESULTS_DIR,
        "evaluation_dashboard.png"
    )

    plt.savefig(
        save_path,
        dpi=300
    )

    plt.show()

# =========================================================
# PHASE DASHBOARD
# =========================================================

def generate_phase_dashboard(
    phase1_history,
    phase2_history
):

    fig = plt.figure(figsize=(18, 10))

    # PHASE 1

    plt.subplot(2,3,1)

    plt.plot(
        phase1_history["train_acc"],
        label="Train"
    )

    plt.plot(
        phase1_history["val_acc"],
        label="Validation"
    )

    plt.title("Phase 1")

    plt.legend()

    plt.grid(True)

    # PHASE 2

    plt.subplot(2,3,2)

    plt.plot(
        phase2_history["train_acc"],
        label="Train"
    )

    plt.plot(
        phase2_history["val_acc"],
        label="Validation"
    )

    plt.title("Phase 2")

    plt.legend()

    plt.grid(True)

    # COMPARISON

    plt.subplot(2,3,3)

    plt.plot(
        phase1_history["val_acc"],
        label="Transfer Learning"
    )

    phase2_epochs = range(
        len(phase1_history["val_acc"]),
        len(phase1_history["val_acc"])
        + len(phase2_history["val_acc"])
    )

    plt.plot(
        phase2_epochs,
        phase2_history["val_acc"],
        label="Fine-Tuning"
    )

    plt.title("Validation Accuracy Comparison")

    plt.legend()

    plt.grid(True)

    # F1

    plt.subplot(2,3,4)

    plt.plot(
        phase1_history["f1_score"],
        label="Transfer Learning"
    )

    plt.plot(
        phase2_epochs,
        phase2_history["f1_score"],
        label="Fine-Tuning"
    )

    plt.title("F1 Comparison")

    plt.legend()

    plt.grid(True)

    # LOSS

    plt.subplot(2,3,5)

    plt.plot(
        phase1_history["val_loss"],
        label="Transfer Learning"
    )

    plt.plot(
        phase2_epochs,
        phase2_history["val_loss"],
        label="Fine-Tuning"
    )

    plt.title("Loss Comparison")

    plt.legend()

    plt.grid(True)

    plt.tight_layout()

    save_path = os.path.join(
        Config.RESULTS_DIR,
        "phase_dashboard.png"
    )

    plt.savefig(
        save_path,
        dpi=300
    )

    plt.show()

# =========================================================
# MAIN
# =========================================================

def main():

    class_names = load_classes()

    val_loader = load_validation_loader()

    model = load_model(
        len(class_names)
    )

    all_preds, all_labels = run_inference(
        model,
        val_loader
    )

    metrics = calculate_metrics(
        all_labels,
        all_preds,
        class_names
    )

    history, phase1_history, phase2_history = load_histories()

    generate_dashboard(
        metrics,
        history
    )

    generate_phase_dashboard(
        phase1_history,
        phase2_history
    )

    print("\nEvaluation Complete")

    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")

# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    main()

Device: cuda
GPU: Tesla T4
Loaded classes: 38

Loading dataset...
Validation samples: 10861

Loading model...


FileNotFoundError: [Errno 2] No such file or directory: '/content/efficientnet_b3_final.pth'